<center>
  <h1>Chat With Pdf</h1>
    Deep Learning Project
</center>


<h2>Import Libraries</h2>

In [ ]:
 pip install PyPDF2

In [ ]:
from PyPDF2 import PdfReader

<h2>Data quality analysis</h2>

In [ ]:
reader = PdfReader("/content/Next.pdf")
print("Number of pages = ",len(reader.pages))

Number of pages =  94


In [ ]:
# inspect individual page
page0 = reader.pages[0]
print(page0.extract_text()[:500])

 
Prof. Arjun V. Bala  NextJS  Page 1 of 94 
 Introduction to Next.js  ................................ ................................ ................................ ...............  3 
What is Next.js?  ................................ ................................ ................................ .....................  3 
Main features of Next.js  ................................ ................................ ................................ .........  3 
History of Next.js  ........


In [ ]:
# check empty page
# enumerate() is a built-in Python function that lets you loop with an index (counter) and a value at the same time.
empty_page =[]

for i,page in enumerate(reader.pages):
    if not page.extract_text():
        empty_page.append(i)

print("Empty Pages : ",empty_page)

Empty Pages :  []


In [ ]:
# Full text Extraction
text = ""

for page in reader.pages:
    if page.extract_text():
        text+=page.extract_text()

len(text)

131148

In [ ]:
# get pdf metadata
metadata = reader.metadata
print("Title:", metadata.title)
print("Author:", metadata.author)
print("Creator:", metadata.creator)
print("Producer:", metadata.producer)

Title: None
Author: arjun bala
Creator: Microsoft® Word 2021
Producer: PDFill PDF Editor 15.0


In [ ]:
# Extract text for NLP

full_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += text + "\n"

print(full_text[:1000])  # preview


 
Prof. Arjun V. Bala  NextJS  Page 1 of 94 
 Introduction to Next.js  ................................ ................................ ................................ ...............  3 
What is Next.js?  ................................ ................................ ................................ .....................  3 
Main features of Next.js  ................................ ................................ ................................ .........  3 
History of Next.js  ................................ ................................ ................................ ....................  3 
Creating first Next.js application  ................................ ................................ ............................  3 
Routing Fundamentals  ................................ ................................ ................................ ...............  7 
Pages  ................................ ................................ ................................ ..

In [ ]:
page = reader.pages[0]

print(type(page))
print(page.keys())


<class 'PyPDF2._page.PageObject'>
dict_keys(['/StructParents', '/Type', '/Contents', '/Tabs', '/MediaBox', '/Parent', '/Resources', '/Group', '/Annots'])


In [ ]:
# extracting image
page = reader.pages[0]

for img in page.images:
    with open(img.name, "wb") as f:
        f.write(img.data)


<h1>Splitting Data Using LangChain</h1>

In [ ]:
!pip install langchain


In [ ]:
!pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 300,chunk_overlap = 0)

In [ ]:
chunks = splitter.split_text(full_text)
len(chunks)

498

In [ ]:
chunks[0]

'Prof. Arjun V. Bala  NextJS  Page 1 of 94 \n Introduction to Next.js  ................................ ................................ ................................ ...............  3'

In [ ]:
chunks[1]

'What is Next.js?  ................................ ................................ ................................ .....................  3 \nMain features of Next.js  ................................ ................................ ................................ .........  3'

In [ ]:
print(len(chunks)) # This will show the actual number of chunks
# chunks[1849] # This line will still cause an error as 1849 is out of range for a list of length 498

498


In [ ]:
splitter_overlap = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap=50)
chunks_overlap = splitter_overlap.split_text(text)

In [ ]:
chunks[0][-500:]

'Prof. Arjun V. Bala  NextJS  Page 1 of 94 \n Introduction to Next.js  ................................ ................................ ................................ ...............  3'

In [ ]:
chunks_overlap[0][-50]

's'

<h2>HuggingFace Embedding</h2>

In [ ]:
 pip install -U langchain langchain-community sentence-transformers

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Embbed some sentence
vector = embeddings.embed_query("What is deep learning?")
len(vector)

384

In [ ]:
# vector

In [ ]:
# Compare some sentense
v1 = embeddings.embed_query("CNN is used for image processing")
v2 = embeddings.embed_query("Convolutional neural networks process images")
v3 = embeddings.embed_query("Football is a sport")

In [ ]:
# np.linalg.norm(a) - This calculates the length of a vector.
# np.dot - Multiplies matching numbers and adds them

import numpy as np

def cosine_similarity(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

print("v1 vs v2:", cosine_similarity(v1, v2))
print("v1 vs v3:", cosine_similarity(v1, v3))

v1 vs v2: 0.5678051321630558
v1 vs v3: 0.05757565855213151


In [ ]:
 !pip install chromadb
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings
)

ImportError: Could not import chromadb python package. Please install it with `pip install chromadb`.

In [ ]:
vector_db

<h2>Chroma Db</h2>

In [ ]:
 pip install chromadb

In [ ]:
from langchain_community.vectorstores import Chroma

In [ ]:
db = Chroma.from_texts(texts=chunks_overlap,embedding=embeddings)

In [ ]:
# Search similiar text
results = db.similarity_search("routing in next js",k=3)
results

[Document(metadata={}, page_content='There are four ways to navigate between routes in Next.js:  \n• Using the <Link>  Component  \n• Using the useRouter  hook (Client Components)  \n• Using the redirect  function (Server Components)  \n• Using the native History  API \nexport default function  Blog() { \n  return ('),
 Document(metadata={}, page_content='Prof. Arjun V. Bala  NextJS  Page 7 of 94 \n Routing Fundamentals  \nNext.js uses file -system based routing, meaning you can use folders and files to define routes.  \nPages  \nA page is UI  (component)  that is rendered on a specific route .'),
 Document(metadata={}, page_content='client -side navigation between routes. It is the primary and recommended way to navigate \nbetween routes in Next.js.  \nYou can use it by importing it from next/link, and passing a href prop to the component:  \n \nThe following props can be passed to the <Link> component:')]

In [ ]:
for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n", r.page_content[:-1])



Result 1:
 There are four ways to navigate between routes in Next.js:  
• Using the <Link>  Component  
• Using the useRouter  hook (Client Components)  
• Using the redirect  function (Server Components)  
• Using the native History  API 
export default function  Blog() { 
  return 

Result 2:
 Prof. Arjun V. Bala  NextJS  Page 7 of 94 
 Routing Fundamentals  
Next.js uses file -system based routing, meaning you can use folders and files to define routes.  
Pages  
A page is UI  (component)  that is rendered on a specific route 

Result 3:
 client -side navigation between routes. It is the primary and recommended way to navigate 
between routes in Next.js.  
You can use it by importing it from next/link, and passing a href prop to the component:  
 
The following props can be passed to the <Link> component


<h2>Ollama</h2>

In [ ]:
!pip install langchain-ollama

In [ ]:
from langchain_ollama.llms import OllamaLLM

In [ ]:
import subprocess
import time
import os

print("Starting Ollama setup...")

# Define the ollama executable path
ollama_executable = '/usr/local/bin/ollama'

# --- Ensure zstd is installed (dependency for Ollama) ---
try:
    print("Checking/Installing zstd...")
    subprocess.run(["apt-get", "update"], check=True, capture_output=True)
    subprocess.run(["apt-get", "install", "-y", "zstd"], check=True, capture_output=True)
    print("zstd installed/updated successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error installing zstd: {e.stderr.decode()}")
except FileNotFoundError:
    print("apt-get not found. Skipping zstd installation (not a Debian-based system?).")

# --- Install Ollama if not present or verify version ---
if not os.path.exists(ollama_executable):
    print(f"Ollama binary not found at {ollama_executable}. Installing Ollama...")
    try:
        # Download the installation script
        install_script_url = "https://ollama.com/install.sh"
        download_command = ["curl", "-fsSL", install_script_url]
        script_content = subprocess.run(download_command, check=True, capture_output=True, text=True).stdout

        # Execute the installation script
        # Using 'sh' with input to run the script content
        process = subprocess.run(["sh"], input=script_content, check=True, capture_output=True, text=True)
        print("Ollama installation output:")
        print(process.stdout)
        print(process.stderr)
        print("Ollama installation script executed successfully.")

    except subprocess.CalledProcessError as e:
        print(f"Error during Ollama installation: {e}")
        print(f"Stdout: {e.stdout}")
        print(f"Stderr: {e.stderr}")
    except Exception as e:
        print(f"An unexpected error occurred during Ollama installation: {e}")
else:
    print(f"Ollama binary found at {ollama_executable}.")
    try:
        version_process = subprocess.run([ollama_executable, '--version'], capture_output=True, text=True)
        print("Ollama version check:")
        print(version_process.stdout.strip())
    except Exception as e:
        print(f"Error checking Ollama version: {e}")

# --- Start Ollama server ---
print("Starting Ollama server...")
try:
    # Attempt to kill any existing Ollama server processes
    subprocess.run(['pkill', '-f', 'ollama serve'], check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("Killed any existing Ollama server processes (if any).")
except Exception as e:
    print(f"Could not kill existing Ollama process: {e}")

# Start the server as a background process
server_process = subprocess.Popen([ollama_executable, 'serve'], preexec_fn=os.setsid, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("Ollama server started. Waiting for it to become ready...")

# Give the server a moment to start up
time.sleep(10) # Increased sleep time for server startup reliability

# --- Pull the llama2 model ---
print("Pulling llama2 model (this may take some time)...")
# Check if model is already pulled to avoid re-downloading
try:
    list_models_process = subprocess.run([ollama_executable, 'list'], check=True, capture_output=True, text=True)
    if "llama2" in list_models_process.stdout:
        print("llama2 model already pulled.")
    else:
        pull_process = subprocess.run([ollama_executable, 'pull', 'llama2'], check=True, capture_output=True, text=True)
        print(pull_process.stdout)
        print(pull_process.stderr)
        print("\nllama2 model pulled successfully. Ollama server should be ready.")
except subprocess.CalledProcessError as e:
    print(f"\nFailed to pull llama2 model: {e.stderr}")
    print("Please check Ollama server status and logs.")
except Exception as e:
    print(f"An unexpected error occurred during model pulling: {e}")

# Initialize the LLM
from langchain_ollama.llms import OllamaLLM
llm = OllamaLLM(model="llama2")
print("OllamaLLM initialized with model 'llama2'.")

Starting Ollama setup...
Checking/Installing zstd...
zstd installed/updated successfully.
Ollama binary not found at /usr/local/bin/ollama. Installing Ollama...
Ollama installation output:

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#=#=#                                                                         
##O#-#                                                                        
##O=#  #                                                                      

                                                                           0.0%
                                                                           0.1%
                                                                           0.2%
                                                                           0.5%
                                                                           0.9%
                                                                           1.2%
#            

In [ ]:
response=llm.invoke("which organization introduce next.js")

In [ ]:
print(response)

Next.js is an open-source platform for building server-rendered, statically generated websites and web applications. It was introduced by the company Vercel, which is a web hosting and platform company that is part of the Procter & Gamble company. Vercel was founded in 2015 and is headquartered in San Francisco, California.

Next.js was introduced in 2016 as a way to simplify the process of building web applications with React, a popular JavaScript library for building user interfaces. The platform provides a set of features and tools that make it easier to build and deploy web applications, including automatic code splitting, server-side rendering, and support for static site generation.

Next.js is designed to work seamlessly with React, and it provides a set of APIs and tools that make it easy to build complex web applications. It also integrates with a number of other popular development tools and services, including GitHub, AWS, and Google Cloud Platform.

Some of the key features

In [ ]:
context = results[0].page_content

prompt = f"""You are an honest assistant.
        You will accept PDF files and you will answer the question asked by the user appropriately.
        If you don't know the answer, just say you don't know. Don't try to make up an answer.

        Context:
        {context}

        Question:
        what is next js?
        """

print(llm.invoke(prompt))


Thank you for asking! Next.js is a popular React-based framework for building server-rendered, statically generated websites and web applications. It provides a set of features and tools to make it easier to build and deploy web applications with a server-side rendering (SSR) approach.

SSR means that the initial HTML response from the server is generated using server-side code, and the client-side code is used to update the page dynamically based on user interactions. This approach can provide better SEO, faster page loads, and a smoother user experience.

Next.js provides a number of features, including:

* Automatic code splitting and optimization for better performance
* Built-in support for server-side rendering (SSR) and client-side rendering (C SR)
* Easy integration with popular APIs and services, such as API Gateway and Amazon DynamoDB
* Support for popular routing libraries, such as React Router
* Built-in support for internationalization (i18n) and localization (L10n)

Overa

<h2>RetrivalQa</h2>

In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

In [ ]:
retriever


VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x788417c3fd70>, search_kwargs={'k': 5})

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are an honest assistant.
You will accept PDF files and you will answer the question asked by the user appropriately.
If you don't know the answer, just say you don't know. Don't try to make up an answer.

Context:
{context}

Question:
{question}
""")

In [ ]:
from langchain_core.runnables import RunnablePassthrough

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:
rag_chain = (
    {
        "context" : retriever | format_docs,
        "question" : RunnablePassthrough()
    }
    | prompt
    | llm
)

In [ ]:
rag_chain.invoke("what is next js?")

'Next.js is a React framework for building full-stack web applications. It allows developers to use React components to build user interfaces and provides additional features and optimizations through its underlying technology. Next.js also automates and configures tooling needed for React, making it easier to develop and deploy web applications.'

# Task
Install Ollama, pull the 'llama2' model, and start the Ollama server to make it available for `langchain-ollama`.

## Install Ollama

### Subtask:
Download and run the Ollama installation script to set up the Ollama binary in the Colab environment.


**Reasoning**:
To download and run the Ollama installation script, I will execute the provided curl command in a code cell.



In [ ]:
# import subprocess

# try:
#     # Download and run the Ollama installation script
#     install_command = "curl -fsSL https://ollama.com/install.sh | sh"
#     process = subprocess.run(install_command, shell=True, check=True, capture_output=True, text=True)
#     print("Ollama installation output:")
#     print(process.stdout)
#     print(process.stderr)
#     print("Ollama installation script executed successfully.")

#     # Verify Ollama installation
#     print("\nVerifying Ollama installation...")
#     verify_command = "ollama --version"
#     verify_process = subprocess.run(verify_command, shell=True, check=True, capture_output=True, text=True)
#     print("Ollama version:")
#     print(verify_process.stdout)
#     print("Ollama verification successful.")

# except subprocess.CalledProcessError as e:
#     print(f"Error during Ollama installation/verification: {e}")
#     print(f"Stdout: {e.stdout}")
#     print(f"Stderr: {e.stderr}")
# except FileNotFoundError:
#     print("Error: 'curl' or 'ollama' command not found. Ensure they are installed and in your PATH.")
# except Exception as e:
#     print(f"An unexpected error occurred: {e}")

**Reasoning**:
The previous attempt to install Ollama failed because `zstd` was not found. I need to install `zstd` first, which is a dependency for Ollama installation. Since this is a Colab environment, `apt-get` will be used.



In [ ]:
# import subprocess

# try:
#     # Install zstd
#     print("Installing zstd...")
#     subprocess.run("sudo apt-get update", shell=True, check=True, capture_output=True, text=True)
#     zstd_install_command = "sudo apt-get install -y zstd"
#     process = subprocess.run(zstd_install_command, shell=True, check=True, capture_output=True, text=True)
#     print("zstd installation output:")
#     print(process.stdout)
#     print(process.stderr)
#     print("zstd installed successfully.")

#     # Download and run the Ollama installation script again
#     print("\nRetrying Ollama installation...")
#     install_command = "curl -fsSL https://ollama.com/install.sh | sh"
#     process = subprocess.run(install_command, shell=True, check=True, capture_output=True, text=True)
#     print("Ollama installation output:")
#     print(process.stdout)
#     print(process.stderr)
#     print("Ollama installation script executed successfully.")

#     # Verify Ollama installation
#     print("\nVerifying Ollama installation...")
#     verify_command = "ollama --version"
#     verify_process = subprocess.run(verify_command, shell=True, check=True, capture_output=True, text=True)
#     print("Ollama version:")
#     print(verify_process.stdout)
#     print("Ollama verification successful.")

# except subprocess.CalledProcessError as e:
#     print(f"Error during installation/verification: {e}")
#     print(f"Stdout: {e.stdout}")
#     print(f"Stderr: {e.stderr}")
# except FileNotFoundError:
#     print("Error: 'curl', 'ollama', or 'apt-get' command not found. Ensure they are installed and in your PATH.")
# except Exception as e:
#     print(f"An unexpected error occurred: {e}")